In [1]:
import numpy as np
from FallbackGen import FallbackGen
from TDECalculator import TDECalculator
import gc

In [2]:
MBH = 1e6
Rp = 25
a = -0.2
N = 5000
E = 1.0
Q = 0.0
orbit="rel"

In [3]:
mass_r = TDECalculator('MAMS2Msun', orbit, MBH, Rp, a, N=N)

In [4]:
sample_r = mass_r.rel_whole_star_sample()

In [5]:
radii_r = sample_r['rr']

rtde = mass_r.R_TDE
Lz = mass_r.mom_kerr_analytic(Rp, a)
E = 1.0
Q = 0

radii = np.where(
    radii_r <= 0.5,
    rtde - radii_r * mass_r.Rstar,
    rtde + radii_r * mass_r.Rstar
)

deltaE = mass_r.Rstar / mass_r.Rp**2 

i = int(np.random.uniform(0, 1132))
j = int(np.random.uniform(0, 90000))
dE = sample_r['dEnergy_random'] * deltaE 
dLz = mass_r.dLz_random
dQ = mass_r.dQ_random
mass_ratio = mass_r.mass_ratio

In [6]:
dT_r = FallbackGen(mass_ratio, a, radii, Rp, E, Lz, Q, dE, dLz, dQ, N)
np.save('dT_r.npy', dT_r.dTs)
gc.collect()

Computing radial periods for 116,280,000 particles ...
  E  range: [0.935909, 1.064101]
  Q  range: [-7.575e-03, 7.576e-03]
  chunk_size = 50,000
  Finding roots (chunked eigensolver) ...
  roots chunk 11649/11649 (100%)
  Bound: 58,244,445 / 116,280,000
  Valid roots: 57,387,810
  Valid Lambda_r: 57,387,810
  Quadrature: 1148 chunks ...
    chunk 1148/1148  (100%)
  Successful T_r: 57,387,810 / 116,280,000


20

In [7]:
def make_plot_dicts(whole_star_sample, dT, delta):
    dE_rand = whole_star_sample["dEnergy_random"]
    dT_rand = dT / delta
    dMass = whole_star_sample["dMass"]

    bins_E = np.linspace(-2.0, 2.0, 1000)
    bins_T = np.logspace(0.0, 6.0, 1000)

    total_mass = np.sum(dMass)
    if not np.isfinite(total_mass) or total_mass <= 0.0:
        raise ValueError("Sample has non-finite or non-positive total mass")

    valid_E = (
        np.isfinite(dE_rand)
        & np.isfinite(dMass)
        & (dE_rand >= bins_E[0])
        & (dE_rand <= bins_E[-1])
    )
    mass_E, edges_E = np.histogram(
        dE_rand[valid_E],
        bins=bins_E,
        weights=dMass[valid_E] / total_mass,
        density=False,
    )
    hist_E = mass_E / np.diff(edges_E)

    valid_T = (
        np.isfinite(dT_rand)
        & np.isfinite(dE_rand)
        & np.isfinite(dMass)
        & (dT_rand >= bins_T[0])
        & (dT_rand <= bins_T[-1])
    )
    mass_T, edges_T = np.histogram(
        dT_rand[valid_T],
        bins=bins_T,
        weights=dMass[valid_T] / total_mass,
        density=False,
    )
    hist_T = mass_T / np.diff(edges_T)

    energy_fraction = np.sum(dMass[valid_E]) / total_mass
    returning_fraction = np.sum(dMass[valid_T]) / total_mass
    assert np.isclose(
        np.sum(hist_E * np.diff(edges_E)), energy_fraction, rtol=1e-12
    )
    assert np.isclose(
        np.sum(hist_T * np.diff(edges_T)), returning_fraction, rtol=1e-12
    )

    print(
        "mass fractions: "
        f"energy range={energy_fraction:.6f}, "
        f"returning/time range={returning_fraction:.6f}"
    )

    energy_plot = {
        "x": 0.5 * (edges_E[:-1] + edges_E[1:]),
        "y": hist_E,
    }
    fallback_plot = {
        "x": 0.5 * (edges_T[:-1] + edges_T[1:]),
        "y": hist_T,
    }
    return energy_plot, fallback_plot

In [8]:
DeltaE = mass_r.Rstar / mass_r.Rp**2
DeltaT = 1 / DeltaE**1.5

In [9]:
rel_r_E, rel_r_T = make_plot_dicts(sample_r, dT_r.dTs, DeltaT)

mass fractions: energy range=1.000000, returning/time range=0.500083


In [10]:
import json

adden = "m2_rp25_a0p2"

with open(f"Fallback_Data/rel_r_e_{adden}.txt", "w") as f:
    json.dump({k: v.tolist() for k, v in rel_r_E.items()}, f)
with open(f"Fallback_Data/rel_r_t_{adden}.txt", "w") as f:
    json.dump({k: v.tolist() for k, v in rel_r_T.items()}, f)